# 總整：英國二手車定價

上完 `modules` 的工具之後才跑這本。對齊 `slides/C5_二手車總整.md`。延續 `projects/project/car_market_eda.ipynb`，不當新比賽。

評分看流程與可解釋，不看榜。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

HERE = Path.cwd().resolve()
if not (HERE / "load_course_data.py").exists():
    found = None
    for p in [HERE, *HERE.parents]:
        cand = p / "data_mining_course" / "capstone"
        if (cand / "load_course_data.py").exists():
            found = cand
            break
        if (p / "load_course_data.py").exists() and p.name == "capstone":
            found = p
            break
    if found is None:
        raise FileNotFoundError("找不到 capstone/load_course_data.py，請把工作目錄設成該資料夾再跑。")
    HERE = found
sys.path.insert(0, str(HERE))

from load_course_data import (
    SNAPSHOT_YEAR,
    load_house_prices,
    load_insurance_like,
    load_taxi_or_cars_for_time,
    load_telco,
    load_unclean_preview,
    load_used_cars,
    make_power_series,
    setup_plotting,
)

setup_plotting()
print("工作目錄：", HERE)


## 5.1　決策還是定價

經銷商要站得住腳的開價理由。EDA 已指出：排量、折舊、品牌分層、高油耗 ≠ 高價（混淆）。


## 5.2　AI 契約（現場可改這個提示詞）

把下面貼給 AI 當領域助理。它提出假設與轉換；你負責 Why、SMART、洩漏。

```
你是英國二手車市場的領域助理，不是自動特徵機器。
任務：幫經銷商解釋「上架價」可能被什麼驅動。
資料欄位只有：model, year, price, transmission, mileage, fuelType, tax, mpg, engineSize, brand。
快照年假設 2020。沒有事故史、沒有選配、price 是上架價不是成交價。
請列出：
1) 5 Why 一條鏈（停在資料量得到的真因）
2) 每個候選特徵是否通過 SMART（量得到、對開價有用、能講人話）
3) 資料沒有、因此不能做的特徵
禁止：一次生出大量交叉項；禁止用測試集統計。
```


## 5.3　5 Why → SMART → 欄位；先切再轉


In [ ]:
cars, note = load_used_cars(sample_n=15000)
print(note)
raw_n = len(cars)
cars = cars.copy()
cars["model"] = cars["model"].astype(str).str.strip()
impossible_year = cars["year"] > SNAPSHOT_YEAR
impossible_engine = cars["engineSize"] <= 0
impossible_price = cars["price"] <= 0
print("未來年", int(impossible_year.sum()), "排量<=0", int(impossible_engine.sum()), "價格<=0", int(impossible_price.sum()))
cars = cars.loc[~impossible_year & ~impossible_engine & ~impossible_price].copy()
cars["car_age"] = SNAPSHOT_YEAR - cars["year"]
cars["log_mileage"] = np.log1p(cars["mileage"])
print("清理後", len(cars), "（從", raw_n, "）")

train, test = train_test_split(cars, test_size=0.2, random_state=42)
train, test = train.copy(), test.copy()
brand_mean = train.groupby("brand")["price"].mean()
gmean = train["price"].mean()
for part in (train, test):
    part["brand_mean_train"] = part["brand"].map(brand_mean).fillna(gmean)

print("品牌均價只來自訓練集。測試集 BMW 均價特徵：", float(test.loc[test["brand"] == "BMW", "brand_mean_train"].iloc[0]))
print("訓練集 BMW 均價：", float(brand_mean["BMW"]))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

num_cols = ["car_age", "log_mileage", "engineSize", "mpg", "tax", "brand_mean_train"]
cat_cols = ["fuelType", "transmission", "brand"]
ytr = train["price"].values
yte = test["price"].values

pre_lin = ColumnTransformer(
    [
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)
lin = Pipeline([("pre", pre_lin), ("model", LinearRegression())])
lin.fit(train[num_cols + cat_cols], ytr)
pred_lin = lin.predict(test[num_cols + cat_cols])
print("線性  MAE", round(mean_absolute_error(yte, pred_lin), 1), "R2", round(r2_score(yte, pred_lin), 3))

# 樹：類別先在訓練摺內編成代碼，避免 LabelEncoder 看測試集未見值時亂掉
train_t = train.copy()
test_t = test.copy()
for col in cat_cols:
    mapping = {k: i for i, k in enumerate(train_t[col].astype(str).unique())}
    train_t[col] = train_t[col].astype(str).map(mapping)
    test_t[col] = test_t[col].astype(str).map(mapping).fillna(-1)

tree_cols = num_cols + cat_cols
hgb = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.08, max_iter=200, random_state=42)
hgb.fit(train_t[tree_cols], ytr)
pred_hgb = hgb.predict(test_t[tree_cols])
print("HGB   MAE", round(mean_absolute_error(yte, pred_hgb), 1), "R2", round(r2_score(yte, pred_hgb), 3))


## 5.4　驗收：係數講人話；重要性對回 Why；mpg 混淆


In [ ]:
ohe = lin.named_steps["pre"].named_transformers_["cat"]
feat_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
coef = lin.named_steps["model"].coef_
coef_tbl = pd.DataFrame({"feature": feat_names, "coef": coef}).sort_values("coef", key=np.abs, ascending=False)
print("線性係數（方向要能辯護：車齡／里程應為負，排量應為正）")
print(coef_tbl.head(12).to_string(index=False))


In [ ]:
try:
    from sklearn.inspection import permutation_importance

    perm = permutation_importance(
        hgb, test_t[tree_cols], yte, n_repeats=5, random_state=42, n_jobs=1
    )
    imp = pd.Series(perm.importances_mean, index=tree_cols).sort_values(ascending=False)
    print("排列重要性（對不回 Why 就可疑）：")
    print(imp)
except Exception as exc:
    print("permutation_importance 略過：", exc)


In [ ]:
print("mpg 與 price 相關（全訓練集）：", round(float(train["mpg"].corr(train["price"])), 3))
print("這是混淆：省油的多半是小車／小排量，不是「愈耗油愈值錢」。")
fig, ax = plt.subplots(figsize=(5, 4))
sample = train.sample(n=min(3000, len(train)), random_state=0)
ax.scatter(sample["mpg"], sample["price"], s=6, alpha=0.25)
ax.set_xlabel("mpg")
ax.set_ylabel("price")
ax.set_title("高油耗 ≠ 高價")


## 5.5　評分（下課用）

| 欄 | 問自己 |
|---|---|
| 問題 | SCQA／Why 清不清楚 |
| 洩漏 | 有無先切再轉、品牌均價是否只看訓練集 |
| 辯護 | 車齡、里程、排量、品牌能否口頭講人話；mpg 有沒有當因果 |

自學：`modules/module_01`–`11`。複本／資料檔：`modules/extension/`。
